# Accessible Technical Concept Explainer

A conversational tool that takes a technical question or concept and explains it in a way that is optimized for screen reader and keyboard-only users — people who cannot rely on diagrams, visual metaphors, or "as you can see" phrasing to understand technical ideas. 
The tool is built twice: once using the Anthropic API for nuanced, context-aware explanations, and once using a local model, Llama 3.2, to demonstrate the same concept running entirely offline without an API key. 
The explanation format is deliberately structured for linear audio consumption: a one-sentence plain language summary first, followed by an analogy that uses no visual references, followed by a step-by-step breakdown using numbered points that a screen reader announces clearly, followed by a concrete real-world example grounded in something a blind or low vision user would encounter in daily life, and ending with one follow-up question the user might want to ask next. 
The tool avoids spatial metaphors ("as shown above," "the box on the left," "the diagram below"), color references used as the sole means of conveying information, and ASCII art or table-based layouts that become noise when read aloud.

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI


In [ ]:
# Load environment variables

load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

# Check the key

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")


Anthropic API Key exists and begins sk-ant-


In [3]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
system_prompt = """
You are a technical educator specializing in explaining complex concepts 
to blind and low vision learners who use screen readers and keyboard-only 
navigation. Your explanations are optimized for audio consumption — 
they will be read aloud by a screen reader, not read visually.

Follow these rules on every response:

STRUCTURE — always use this exact order:
1. One-sentence plain language summary of the concept
2. An analogy that uses no visual references — 
   base it on sound, touch, movement, sequence, or 
   everyday non-visual experience
3. Step-by-step breakdown — use numbered steps, 
   each step one sentence, so the screen reader 
   announces them as a clear sequence
4. A concrete example grounded in something a blind or 
   low vision user encounters in daily life — 
   VoiceOver navigation, Braille display output, 
   keyboard shortcuts, accessible app behavior
5. One natural follow-up question the learner might want 
   to explore next

LANGUAGE RULES — never use:
- Visual spatial metaphors: "as shown above", "the box on the left", 
  "the diagram below", "as you can see", "looking at this"
- Color as the sole means of conveying meaning: 
  instead of "the red items are errors" say 
  "items marked as errors"
- ASCII art, tables, or visual layouts that 
  become noise when read aloud — use numbered 
  or bulleted lists instead
- Parenthetical asides that interrupt screen reader flow — 
  place all context in the main sentence

TONE — practitioner to practitioner. 
You are not simplifying for a beginner unless asked. 
You are removing visual assumptions from a technical explanation, 
not dumbing it down. The learner is technically capable — 
they just need an explanation that works without a screen.
"""


In [5]:
user_prompt_template = """
Please explain the following technical concept in a way that is 
fully accessible to a screen reader user. 
Follow your structured explanation format.

Concept or question: {user_question}

If this concept is typically explained using a diagram, 
flowchart, or visual metaphor, explicitly acknowledge that 
and provide an alternative explanation grounded in 
sequence, process, or analogy instead.
"""


In [6]:
def explain_with_anthropic(question):
    """Explain a technical concept using the Anthropic API."""
    response = anthropic.chat.completions.create(
        model="claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":
                user_prompt_template.format(user_question=question)}
        ]
    )
    return response.choices[0].message.content

In [7]:
def explain_with_ollama(question, model="llama3.2"):
    """
    Explain a technical concept using a local Ollama model.
    Runs entirely offline — no API key required.
    """
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":
                user_prompt_template.format(user_question=question)}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def run_explainer():
    """
    Main loop — runs both models on the same question
    so you can compare output quality.
    """
    print("=== Accessible Technical Concept Explainer ===")
    print("Explanations are structured for screen reader consumption.")
    print("Type 'quit' to exit.\n")

    question = input("Enter your technical question: ").strip()

    print("\n--- Anthropic (Claude) response ---\n")
    try:
        anthropic_response = explain_with_anthropic(question)
        print(anthropic_response)
    except Exception as e:
        print(f"Anthropic call failed: {e}")

    print("\n--- Ollama response (local, offline) ---\n")
    try:
        ollama_response = explain_with_ollama(question)
        print(ollama_response)
    except Exception as e:
        print(f"Ollama call failed: {e}")

print("\n" + "="*60 + "\n")

IndentationError: unexpected indent (2670115308.py, line 22)

In [10]:
run_explainer()

=== Accessible Technical Concept Explainer ===
Explanations are structured for screen reader consumption.
Type 'quit' to exit.


--- Anthropic (Claude) response ---

## The Nullish Coalescing Operator in JavaScript

**Plain language summary:** The double question mark operator, written as two consecutive question marks, lets you provide a fallback value that is used only when the left side is null or undefined, and nothing else.

---

**Analogy:** Think of it like a Braille display that shows the content of a cell if any text is stored there, but switches to a default label if that cell is completely empty — not if the cell contains a zero, not if it contains a blank space, only if it contains absolutely nothing at all. Other fallback tools would trigger on a zero or a blank space, but this one is more precise.

---

**Step-by-step breakdown:**

1. JavaScript evaluates the expression on the left side of the operator.
2. If that left side evaluates to null or undefined, JavaScript uses 